# 🤖 Multiple Models + One Combined Comparison

### Step 1 — Load the dataset

In [184]:
import pandas as pd
import numpy as np

df = pd.read_csv("employee_promotion_5000.csv")

In [185]:
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df["Promoted"].value_counts())

   Employee_ID   Age  Experience  Monthly_Bonus Department Education  Promoted
0        50001  56.0        26.0         8270.0        Web  Bachelor         1
1        50002  38.0        15.0         4705.0        NaN    Master         0
2        50003  36.0        15.0         6916.0   Software  Bachelor         0
3        50004  43.0        21.0         4976.0       Data  Bachelor         0
4        50005  54.0        24.0         7571.0      Cloud  Bachelor         0
(5000, 7)
Employee_ID        int64
Age              float64
Experience       float64
Monthly_Bonus    float64
Department        object
Education         object
Promoted           int64
dtype: object
Employee_ID       0
Age              50
Experience       50
Monthly_Bonus    50
Department       75
Education        50
Promoted          0
dtype: int64
Promoted
0    4195
1     805
Name: count, dtype: int64


### Step 2 — Create X and y

In [186]:
X = df[[
    "Age",
    "Experience",
    "Monthly_Bonus",
    "Department",
    "Education"
]]

y = df["Promoted"]

### Step 3 — Define column types

In [187]:
numerical_features = [
    "Age",
    "Experience",
    "Monthly_Bonus"
]

categorical_features = [
    "Department",
    "Education"
]

### Step 4 — Build preprocessing

In [188]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

### Step 5 — Train/Test Split

In [189]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Step 6 — Create ALL models

In [190]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

model = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

### Step 7 — Build a Pipeline for each model

In [191]:
pipelines = {}

for name, model in model.items():
    pipelines[name] = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

### Step 8 — Train ALL models

In [192]:
for name, pipeline in pipelines.items():

    pipeline.fit(
        X_train,
        y_train
    )

    print(name, ": Trained")

Logistic Regression : Trained
Decision Tree : Trained


Random Forest : Trained


### Step 9 — Generate ALL predictions

In [193]:
predictions = {}

for name, pipeline in pipelines.items():

    predictions[name] = pipeline.predict(X_test)

In [194]:
for name, prediction in predictions.items():

    print(name)
    print(prediction[:10])
    print()

Logistic Regression
[0 0 0 0 0 0 0 0 0 0]

Decision Tree
[0 0 0 0 1 0 0 0 0 0]

Random Forest
[0 0 0 0 1 0 0 0 0 0]



### Step 10 — Now evaluate everything together

In [195]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

evaluation_results = []

for name, prediction in predictions.items():
    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_test,
            prediction
        ),
        "Precision": precision_score(
            y_test,
            prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            prediction,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            prediction,
            zero_division=0
        )
    })

In [196]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

print(evaluation_df.sort_values(
    "F1",
    ascending=False
))

                 Model  Accuracy  Precision    Recall        F1
2        Random Forest     0.828   0.396226  0.130435  0.196262
0  Logistic Regression     0.839   0.500000  0.024845  0.047337
1        Decision Tree     0.831   0.250000  0.024845  0.045198


### Step 11 — NOW do Cross-Validation

In [197]:
from sklearn.model_selection import cross_val_score

cv_results = []

for name, pipeline in pipelines.items():

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=5,
        scoring="f1"
    )

    cv_results.append({
        "Model": name,
        "Mean_CV_F1": scores.mean(),
        "STD_CV_F1": scores.std()
    })

In [198]:
cv_df = pd.DataFrame(cv_results)

print(cv_df.sort_values(
    "Mean_CV_F1",
    ascending=False
))

                 Model  Mean_CV_F1  STD_CV_F1
2        Random Forest    0.190699   0.024525
0  Logistic Regression    0.088083   0.021051
1        Decision Tree    0.077229   0.034797


### Step 12 — Final Comparison Table

In [199]:
final_comparison = evaluation_df.merge(
    cv_df,
    on="Model"
)

final_comparison = final_comparison.sort_values(
    "Mean_CV_F1",
    ascending=False
)

print(final_comparison)

                 Model  Accuracy  Precision    Recall        F1  Mean_CV_F1  \
2        Random Forest     0.828   0.396226  0.130435  0.196262    0.190699   
0  Logistic Regression     0.839   0.500000  0.024845  0.047337    0.088083   
1        Decision Tree     0.831   0.250000  0.024845  0.045198    0.077229   

   STD_CV_F1  
2   0.024525  
0   0.021051  
1   0.034797  


In [200]:
best_model_name = final_comparison.loc[
    final_comparison["Mean_CV_F1"].idxmax(),
    "Model"
]

print("Best model:", best_model_name)

Best model: Random Forest


### Train vs Test Performance

In [201]:
train_results = []

for name, pipeline in pipelines.items():

    train_prediction = pipeline.predict(X_train)

    train_results.append({
        "Model": name,
        "Train_F1": f1_score(
            y_train,
            train_prediction,
            zero_division=0
        )
    })

train_df = pd.DataFrame(train_results)

print(train_df)

                 Model  Train_F1
0  Logistic Regression  0.095652
1        Decision Tree  0.168276
2        Random Forest  0.999223


In [202]:
final_comparison = final_comparison.merge(
    train_df,
    on="Model"
)

print(final_comparison)

                 Model  Accuracy  Precision    Recall        F1  Mean_CV_F1  \
0        Random Forest     0.828   0.396226  0.130435  0.196262    0.190699   
1  Logistic Regression     0.839   0.500000  0.024845  0.047337    0.088083   
2        Decision Tree     0.831   0.250000  0.024845  0.045198    0.077229   

   STD_CV_F1  Train_F1  
0   0.024525  0.999223  
1   0.021051  0.095652  
2   0.034797  0.168276  


In [203]:
from sklearn.metrics import confusion_matrix

for name, prediction in predictions.items():

    matrix = confusion_matrix(
        y_test,
        prediction
    )

    print(name)
    print(matrix)
    print()

Logistic Regression
[[835   4]
 [157   4]]

Decision Tree
[[827  12]
 [157   4]]

Random Forest
[[807  32]
 [140  21]]



### Classification Report

In [204]:
from sklearn.metrics import classification_report

for name, prediction in predictions.items():

    print(name)
    print(
        classification_report(
            y_test,
            prediction,
            zero_division=0
        )
    )

Logistic Regression
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       839
           1       0.50      0.02      0.05       161

    accuracy                           0.84      1000
   macro avg       0.67      0.51      0.48      1000
weighted avg       0.79      0.84      0.77      1000

Decision Tree
              precision    recall  f1-score   support

           0       0.84      0.99      0.91       839
           1       0.25      0.02      0.05       161

    accuracy                           0.83      1000
   macro avg       0.55      0.51      0.48      1000
weighted avg       0.75      0.83      0.77      1000

Random Forest
              precision    recall  f1-score   support

           0       0.85      0.96      0.90       839
           1       0.40      0.13      0.20       161

    accuracy                           0.83      1000
   macro avg       0.62      0.55      0.55      1000
weighted avg       0.78   

In [205]:
final_comparison = final_comparison[
    [
        "Model",
        "Train_F1",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Mean_CV_F1",
        "STD_CV_F1"
    ]
]

final_comparison = final_comparison.sort_values(
    "Mean_CV_F1",
    ascending=False
)

print(final_comparison.round(3))

                 Model  Train_F1  Accuracy  Precision  Recall     F1  \
0        Random Forest     0.999     0.828      0.396   0.130  0.196   
1  Logistic Regression     0.096     0.839      0.500   0.025  0.047   
2        Decision Tree     0.168     0.831      0.250   0.025  0.045   

   Mean_CV_F1  STD_CV_F1  
0       0.191      0.025  
1       0.088      0.021  
2       0.077      0.035  


# Practice

In [206]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

rf_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [207]:
rf_prediction = rf_pipeline.predict(X_test)
rf_probability = rf_pipeline.predict_proba(X_test)

In [208]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [209]:
from sklearn.metrics import confusion_matrix

rf_matrix = confusion_matrix(
    y_test,
    rf_prediction
)

print(rf_matrix)

[[807  32]
 [140  21]]


In [212]:
feature_names = rf_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = rf_pipeline.named_steps[
    "model"
].feature_importances_


feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    "Importance",
    ascending=False
)

print(feature_importance_df)

                     Feature  Importance
2         num__Monthly_Bonus    0.440769
0                   num__Age    0.248566
1            num__Experience    0.206952
3         cat__Department_AI    0.016447
6   cat__Department_Software    0.014723
5       cat__Department_Data    0.014565
7        cat__Department_Web    0.013584
4      cat__Department_Cloud    0.012603
8    cat__Education_Bachelor    0.012179
9      cat__Education_Master    0.011049
10        cat__Education_PhD    0.008563
